# ARGUS — Treino do detector YOLO11 (Colab)

Espelho de `training/train_local.py`: usa o **mesmo** `train_config.yaml` e os mesmos
scripts. Use quando a A5000 local não estiver disponível.

**Runtime → Alterar tipo de runtime → GPU (T4/L4/A100).**

Ordem: instalar → obter código → gerar dataset → treinar → métricas → publicar.

In [ ]:
# 1) Código do projeto (clona o repo público)
!git clone --depth 1 https://github.com/Zagari/argus-threat-modeling.git
%cd argus-threat-modeling/argus

In [ ]:
# 2) Dependências (treino) + checagem de GPU
!pip -q install -r training/requirements-train.txt
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 3) Dataset sintético (regenerado aqui — não vai no git). Ajuste --n conforme o tempo.
!python training/synthetic/generate_synthetic.py --n 3000 --out data/synthetic --seed 42
# Com ícones oficiais: suba os .zip e rode antes:
# !python training/icons/fetch_icons.py --aws-zip aws.zip --azure-zip azure.zip --out data/icons
# depois acrescente --icons-dir data/icons ao comando acima.

In [ ]:
# 4) Sanidade (1 época) — confirma que tudo roda antes do treino longo
!python training/train_local.py --epochs 1 --quick --device 0

In [ ]:
# 5) Treino completo (lê training/train_config.yaml)
!python training/train_local.py --device 0

In [ ]:
# 6) Métricas + curvas
import json, glob, os
run = sorted(glob.glob('training/runs/*'), key=os.path.getmtime)[-1]
print('run:', run)
print(json.dumps(json.load(open(f'{run}/metrics.json')), indent=2, ensure_ascii=False))
from IPython.display import Image, display
for p in ['results.png', 'confusion_matrix_normalized.png']:
    fp = f'{run}/{p}'
    if os.path.exists(fp): display(Image(fp))

In [ ]:
# 7) Publicar no HF Hub (best.pt + Model Card). Faça login primeiro.
# from huggingface_hub import notebook_login; notebook_login()
# !python training/publish_hf.py --weights {run}/weights/best.pt --repo SEU_USUARIO/argus-detector --metrics {run}/metrics.json